# JSpace steering demo — replace answer with 'Pairs'

This notebook demonstrates asking the model `What's the capital city of US?` and using a pre-fitted Jacobian lens to steer the model toward the token `Pairs`. Follow the requirements in `walkthrough.ipynb` and ensure you have a fitted lens available (see `walkthrough.md`).

In [23]:
import torch
import transformers
import jlens
from IPython.display import display
from jlens.vis import notebook_iframe
import json, gzip

In [24]:
# Device and dtype selection (adapt if needed)
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch_dtype = torch.bfloat16
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
    torch_dtype = torch.float16
else:
    device = torch.device("cpu")
    torch_dtype = torch.float32
device, torch_dtype

(device(type='mps'), torch.float16)

In [25]:
# Configure these to match your fitted lens/model
MODEL_NAME = "Qwen/Qwen3.5-4B"
LENS_REPO = "neuronpedia/jacobian-lens"
LENS_REVISION = "qwen-n1000"
LENS_FILE = {
    "Qwen/Qwen3.5-4B": "qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt",
    "Qwen/Qwen3.6-27B": "qwen3.6-27b/jlens/Salesforce-wikitext/Qwen3.6-27B_jacobian_lens_n1000.pt",
}[MODEL_NAME]
MODEL_NAME, LENS_REPO, LENS_FILE

('Qwen/Qwen3.5-4B',
 'neuronpedia/jacobian-lens',
 'qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt')

In [19]:
# Load model and tokenizer and wrap with jlens' LensModel
hf_model = transformers.AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch_dtype, trust_remote_code=True).to(device)
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = jlens.from_hf(hf_model, tokenizer)
print('model device:', getattr(model, 'input_device', device))

Loading weights: 100%|██████████| 426/426 [00:00<00:00, 7422.25it/s]


model device: mps:0


In [26]:
# Load the pre-fitted Jacobian lens (from Hub or local path)
lens = jlens.JacobianLens.from_pretrained(LENS_REPO, filename=LENS_FILE, revision=LENS_REVISION)
lens

Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 23831.27it/s]


JacobianLens(d_model=2560, n_prompts=1000, source_layers=[0..30] (31 layers))

In [27]:
# The user prompt we will ask the model
prompt = "What's the capital city of USA?"
print('prompt:', prompt)
# Tokenize for steering call (single example batch)
ids = torch.tensor([tokenizer.encode(prompt, add_special_tokens=False)], dtype=torch.long, device=getattr(model, 'input_device', device))
print('input ids:', ids.tolist())

prompt: What's the capital city of USA?
input ids: [[3710, 579, 279, 6511, 3177, 314, 7052, 30]]


In [28]:
# Choose the target token text you want to insert via JSpace steering
target_text = "Pairs"
tokens = tokenizer.encode(target_text, add_special_tokens=False)
if len(tokens) == 0:
    raise RuntimeError('tokenizer produced no tokens for target_text')
target_token_id = tokens[0]
print('target_text:', target_text, '-> token ids:', tokens)
print('using target_token_id:', target_token_id)

target_text: Pairs -> token ids: [52405]
using target_token_id: 52405


In [ ]:
# Run a steering intervention at default strength and inspect the effect
strength = 0.2
result = lens.steer(model, ids, target_token_id=target_token_id, strength=strength)
print('clean target rank -> steered target rank:', result.clean_target_ranks.item(), '->', result.steered_target_ranks.item())

# Build a comparison visualization (if running in notebook with display)
comparison = jlens.compute_steering_comparison(model, lens, result, last_n_tokens=32, mask_display=True)
page = jlens.build_steering_comparison_page(comparison, title=f"Steering: {target_text}", description="Steering the next-token toward a chosen token (controllability demo).")
display(notebook_iframe(page, height=700))

clean target rank -> steered target rank: 98119 -> 0


In [30]:
# Decode and display clean and steered top tokens and top-5 candidates
clean_top_id = int(result.clean_top_token_ids[0])
steered_top_id = int(result.steered_top_token_ids[0])
clean_top_token = tokenizer.decode([clean_top_id])
steered_top_token = tokenizer.decode([steered_top_id])
print('clean top-1 token:', clean_top_token, '-> id', clean_top_id)
print('steered top-1 token:', steered_top_token, '-> id', steered_top_id)
# Show top-5 tokens from logits
clean_logits = result.clean_logits[0]
steered_logits = result.steered_logits[0]
def topk_tokens(logits, k=5):
    vals, ids = logits.topk(k)
    return [(int(i), tokenizer.decode([int(i)]), float(v)) for v,i in zip(vals.tolist(), ids.tolist())]
print('clean top-5:', topk_tokens(clean_logits,5))
print('steered top-5:', topk_tokens(steered_logits,5))
# Greedy one-step continuation using steered top token
next_token = steered_top_id
input_ids = result.input_ids.cpu()
generated = tokenizer.decode((input_ids[0].tolist() + [int(next_token)]))
print('greedy one-step generated sequence (steered):', generated)

clean top-1 token: 

 -> id 271
steered top-1 token: Pairs -> id 52405
clean top-5: [(271, '\n\n', 20.25), (198, '\n', 19.125), (471, ' -', 17.125), (3437, ' What', 16.5), (318, ' (', 16.125)]
steered top-5: [(52405, 'Pairs', 28.5), (30, '?', 20.375), (11, ',', 17.875), (13139, ' pairs', 17.625), (198, '\n', 16.75)]
greedy one-step generated sequence (steered): What's the capital city of USA?Pairs


In [33]:
# Generate full continuations: clean, clean+CoT, steered-forced, steered+CoT
# Prepare generation input (HF model expects token ids tensor)
input_ids_gen = tokenizer.encode(prompt, return_tensors="pt").to(getattr(hf_model, 'device', device))

# Clean greedy generation
clean_gen_ids = hf_model.generate(input_ids_gen, max_new_tokens=200, do_sample=False)
clean_generated = tokenizer.decode(clean_gen_ids[0], skip_special_tokens=True)
print('Clean generation:', clean_generated)

# Clean Chain-of-Thought prompt generation
cot_prompt = prompt + " Let's think step by step."
cot_input_ids = tokenizer.encode(cot_prompt, return_tensors="pt").to(getattr(hf_model, 'device', device))
cot_gen_ids = hf_model.generate(cot_input_ids, max_new_tokens=200, do_sample=False)
cot_generated = tokenizer.decode(cot_gen_ids[0], skip_special_tokens=True)
print('Clean CoT generation:', cot_generated)

# Steered forced continuation: append the steered top token, then generate further tokens
forced_ids = input_ids_gen.clone()
forced_ids = torch.cat([forced_ids, torch.tensor([[int(steered_top_id)]], device=forced_ids.device)], dim=1)
steered_cont_ids = hf_model.generate(forced_ids, max_new_tokens=200, do_sample=False)
steered_cont = tokenizer.decode(steered_cont_ids[0], skip_special_tokens=True)
print('Steered forced continuation:', steered_cont)

# Steered + CoT: after forced token append CoT instruction and generate
forced_cot_text = tokenizer.decode(forced_ids[0]) + " Let's think step by step."
forced_cot_ids = tokenizer.encode(forced_cot_text, return_tensors="pt").to(getattr(hf_model, 'device', device))
forced_cot_gen_ids = hf_model.generate(forced_cot_ids, max_new_tokens=200, do_sample=False)
forced_cot_gen = tokenizer.decode(forced_cot_gen_ids[0], skip_special_tokens=True)
print('Steered + CoT generation:', forced_cot_gen)


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Clean generation: What's the capital city of USA?

<think>
Thinking Process:

1.  **Identify the core question:** The user is asking for the capital city of the USA (United States of America).

2.  **Retrieve knowledge:** Access general knowledge about the United States of America.
    *   Country: United States of America (USA)
    *   Capital City: Washington, D.C. (District of Columbia)

3.  **Formulate the answer:** State the capital city clearly.
    *   Draft: The capital city of the USA is Washington, D.C.

4.  **Review and refine:** Is there any ambiguity? No. Is it concise? Yes.
    *   Final Answer: Washington, D.C.

5.  **Output:** Washington, D.C.cw
</think>

The capital city of the USA is **Washington, D.C.** (District of Columbia).


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Clean CoT generation: What's the capital city of USA? Let's think step by step.

<think>
Thinking Process:

1.  **Analyze the Request:**
    *   Question: "What's the capital city of USA?"
    *   Constraint: "Let's think step by step."
    *   Goal: Provide the correct answer while demonstrating a step-by-step reasoning process.

2.  **Identify the Core Fact:**
    *   The question asks for the capital city of the United States of America (USA).
    *   Common knowledge: Washington, D.C.

3.  **Formulate the Step-by-Step Reasoning:**
    *   Step 1: Identify the country in question (United States of America).
    *   Step 2: Recall or verify the definition of a capital city (the city where the government, specifically the executive, legislative, and judicial branches, are located).
    *   Step 3: Retrieve knowledge about the specific capital of the USA.
    *  


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Steered forced continuation: What's the capital city of USA?Pairs: 1. 1. 2. 2. 3. 3. 4. 4. 5. 5. 6. 6. 7. 7. 8. 8. 9. 9. 10. 10. 11. 11. 12. 12. 13. 13. 14. 14. 15. 15. 16. 16. 17. 17. 18. 18. 19. 19. 20. 20. 21. 21. 22. 22. 23. 23. 24. 24. 25. 25. 26. 26. 27. 27. 
Steered + CoT generation: What's the capital city of USA?Pairs Let's think step by step.

<think>
Thinking Process:

1.  **Analyze the Request:**
    *   Question: "What's the capital city of USA?"
    *   Constraint: "Pairs Let's think step by step." (This seems to be a prompt engineering instruction or a specific format request, possibly indicating a need for a step-by-step reasoning process, though "Pairs" might be a typo or a specific model instruction I need to adhere to. Given the context of "Let's think step by step", it's likely a trigger for Chain of Thought reasoning).
    *   Goal: Provide the correct answer with a step-by-step explanation.

2.  **Identify the Core Fact:**
    *   Country: United States of America

## Comparison: Before vs After Steering

The cell below prints the clean (before steering) and steered (after steering) outputs side-by-side so you can compare top tokens, top-5 lists, greedy continuations, and CoT generations.

In [34]:
# Side-by-side comparison: clean vs steered
print('=== Top-1 tokens ===')
print('Clean :', clean_top_token, f'(id {clean_top_id})')
print('Steered:', steered_top_token, f'(id {steered_top_id})')

print('=== Top-5 tokens (clean) ===')
for i, tok, score in topk_tokens(clean_logits,5):
    print(f'{i}: {tok} (score={score:.4f})')

print('=== Top-5 tokens (steered) ===')
for i, tok, score in topk_tokens(steered_logits,5):
    print(f'{i}: {tok} (score={score:.4f})')

print('=== Greedy continuations ===')
print('Clean generation:', clean_generated)
print('Steered forced continuation (forced steered token then generate):', steered_cont)

print('=== Chain-of-Thought (CoT) variants ===')
print('Clean CoT generation:', cot_generated)
print('Steered + CoT generation:', forced_cot_gen)

=== Top-1 tokens ===
Clean : 

 (id 271)
Steered: Pairs (id 52405)
=== Top-5 tokens (clean) ===
271: 

 (score=20.2500)
198: 
 (score=19.1250)
471:  - (score=17.1250)
3437:  What (score=16.5000)
318:  ( (score=16.1250)
=== Top-5 tokens (steered) ===
52405: Pairs (score=28.5000)
30: ? (score=20.3750)
11: , (score=17.8750)
13139:  pairs (score=17.6250)
198: 
 (score=16.7500)
=== Greedy continuations ===
Clean generation: What's the capital city of USA?

<think>
Thinking Process:

1.  **Identify the core question:** The user is asking for the capital city of the USA (United States of America).

2.  **Retrieve knowledge:** Access general knowledge about the United States of America.
    *   Country: United States of America (USA)
    *   Capital City: Washington, D.C. (District of Columbia)

3.  **Formulate the answer:** State the capital city clearly.
    *   Draft: The capital city of the USA is Washington, D.C.

4.  **Review and refine:** Is there any ambiguity? No. Is it concise? Yes.


### Notes
- `lens.steer` performs the intervention using the normalized row of `W_U J_l` internally; we selected the first token id of the tokenized `Pairs` text as the steering target.
- If `Pairs` tokenizes to multiple tokens, steering only the first subtoken is a simple demonstration; for multi-token targets you can steer sequentially or adapt the method.
- Adjust `strength` to make the intervention weaker or stronger; large values may push the model outside the linear approximation.
- Ensure `LENS_FILE` matches your model exactly (same `d_model`, layer count).